# Visualize

In [76]:
!pip install networkx
!pip install graphviz
!pip install pyvis
!pip install torch

In [174]:
import networkx as nx
import pyvis
from pyvis.network import Network
from graphviz import Digraph


def show_graph_interactive(self, filename="graph.html"):
    nodes = {} # Diccionario para almacenar nodos por ID
    edges = []

    def build(v):
        if id(v) not in nodes: #Usamos el id del objeto como key
            nodes[id(v)] = {"label": f"{v.name} | data={v.data:.2f} | grad={v.grad:.2f}", "shape": "box"}
            if v._op: # Si tiene una operación, crea el nodo de operación
                op_node_id = f"op_{id(v)}" #ID unico para el nodo de operacion
                nodes[op_node_id] = {"label": v._op, "shape": "circle", "color": "lightblue", "size": 20} #Nodo de operacion
                for child in v._prev:
                    edges.append((id(child), op_node_id)) #Arista desde los hijos a la operacion
                edges.append((op_node_id, id(v))) #Arista desde la operacion al resultado
                for child in v._prev:
                    build(child) #Llamada recursiva para los hijos

    build(self)

    graph = nx.DiGraph()
    for node_id, node_data in nodes.items():
        graph.add_node(node_id, **node_data) #Agrega los nodos al grafo con sus atributos
    for edge in edges:
        graph.add_edge(edge[0], edge[1], arrows={'to': True, 'from': False})

    net = Network(notebook=True, cdn_resources='in_line', directed=True)

    net.from_nx(graph)
    net.prep_notebook()
    net.show(filename)

def trace(root):
    nodes, edges = set(), set()
    def build(v):
        if v not in nodes:
            nodes.add(v)
            for child in v._prev:
                edges.add((child, v))
                build(child)
    build(root)
    return nodes, edges

def show_graph(root, format='svg', rankdir='LR'):
    """
    format: png | svg | ...
    rankdir: TB (top to bottom graph) | LR (left to right)
    """
    assert rankdir in ['LR', 'TB']
    nodes, edges = trace(root)
    dot = Digraph(format=format, graph_attr={'rankdir': rankdir}) #, node_attr={'rankdir': 'TB'})

    for n in nodes:
        dot.node(name=str(id(n)), label = "{%s | data %.4f | grad %.4f }" % (n.name, n.data, n.grad), shape='record')
        if n._op:
            dot.node(name=str(id(n)) + n._op, label=n._op)
            dot.edge(str(id(n)) + n._op, str(id(n)))

    for n1, n2 in edges:
        dot.edge(str(id(n1)), str(id(n2)) + n2._op)

    return dot


# Modificacion del Value

In [62]:
import math
import numpy as np

# Se define la clase Value
class Value:
    """ stores a single scalar value and its gradient """

    def __init__(self, data, _children=(), _op='', name=''):
        self.data = data
        self.grad = 0
        self.name = name
        # internal variables used for autograd graph construction
        self._backward = lambda: None
        self._prev = set(_children)
        self._op = _op # the op that produced this node, for graphviz / debugging / etc

    def __add__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data + other.data, (self, other), '+')

        def _backward():
            self.grad += out.grad
            other.grad += out.grad
        out._backward = _backward

        return out

    def __mul__(self, other):
        other = other if isinstance(other, Value) else Value(other)
        out = Value(self.data * other.data, (self, other), '*')

        def _backward():
            self.grad += other.data * out.grad
            other.grad += self.data * out.grad
        out._backward = _backward

        return out

    def __pow__(self, other):
        assert isinstance(other, (int, float)), "only supporting int/float powers for now"
        out = Value(self.data**other, (self,), f'**{other}')

        def _backward():
            self.grad += (other * self.data**(other-1)) * out.grad
        out._backward = _backward

        return out

    def relu(self):
        out = Value(0. if self.data < 0 else self.data, (self,), 'ReLU')

        def _backward():
            self.grad += (out.data > 0) * out.grad
        out._backward = _backward

        return out

    def exp(self):
        x = self.data
        out = Value(math.exp(x), (self, ), f'e^{self.data}')

        def _backward():
          self.grad += out.data * out.grad
        out._backward = _backward

        return out


    def tanh(self):
        x = self.data
        t = (np.e ** (2*x) - 1) / (np.e ** (2*x) + 1)
        out = Value(t, _children=(self,), _op='tanh')

        def _backward():
            self.grad += (1 - t**2) * out.grad

        out._backward = _backward
        return out


    def sigmoid(self):
        x = self.data
        s = 1 / (1 + np.e ** (-x))
        out = Value(s, _children=(self,), _op='sigmoid')

        def backward():
            self.grad += s * (1 - s) * out.grad

        out._backward = backward
        return out


    def backward(self):

        # topological order all of the children in the graph
        topo = []
        visited = set()
        def build_topo(v):
            if v not in visited:
                visited.add(v)
                for child in v._prev:
                    build_topo(child)
                topo.append(v)
        build_topo(self)

        # go one variable at a time and apply the chain rule to get its gradient
        self.grad = 1
        for v in reversed(topo):
            v._backward()

    def __neg__(self): # -self
        return self * -1

    def __radd__(self, other): # other + self
        return self + other

    def __sub__(self, other): # self - other
        return self + (-other)

    def __rsub__(self, other): # other - self
        return other + (-self)

    def __rmul__(self, other): # other * self
        return self * other

    def __truediv__(self, other): # self / other
        return self * other**-1

    def __rtruediv__(self, other): # other / self
        return other * self**-1

    def __repr__(self):
        return f"Value(data={self.data}, grad={self.grad}, name={self.name})"


# Prueba del nuevo Value con pytorch


In [249]:
# Pasamos el test
import torch

def test_sanity_check():
    print("Testing")
    x = Value(-4.0)
    z = 2 * x + 2 + x
    q = z.relu() + z * x
    h = (z * z).relu()
    y = h + q + q * x
    y.backward()
    xmg, ymg = x, y

    x = torch.Tensor([-4.0]).double()
    x.requires_grad = True
    z = 2 * x + 2 + x
    q = z.relu() + z * x
    h = (z * z).relu()
    y = h + q + q * x
    y.backward()
    xpt, ypt = x, y

    # forward pass went well
    assert ymg.data == ypt.data.item()
    # backward pass went well
    assert xmg.grad == xpt.grad.item()

def test_more_ops():

    a = Value(-4.0)
    b = Value(2.0)
    c = a + b
    d = a * b + b**3
    c += c + 1
    c += 1 + c + (-a)
    d += d * 2 + (b + a).relu()
    d += 3 * d + (b - a).relu()
    e = c - d
    f = e**2
    g = f / 2.0
    g += 10.0 / f
    g.backward()
    amg, bmg, gmg = a, b, g

    a = torch.Tensor([-4.0]).double()
    b = torch.Tensor([2.0]).double()
    a.requires_grad = True
    b.requires_grad = True
    c = a + b
    d = a * b + b**3
    c = c + c + 1
    c = c + 1 + c + (-a)
    d = d + d * 2 + (b + a).relu()
    d = d + 3 * d + (b - a).relu()
    e = c - d
    f = e**2
    g = f / 2.0
    g = g + 10.0 / f
    g.backward()
    apt, bpt, gpt = a, b, g

    tol = 1e-6
    # forward pass went well
    assert abs(gmg.data - gpt.data.item()) < tol
    # backward pass went well
    assert abs(amg.grad - apt.grad.item()) < tol
    assert abs(bmg.grad - bpt.grad.item()) < tol

In [127]:
test_sanity_check()

Testing


In [128]:
test_more_ops()

# Modificación de `nn.py`

In [257]:
import random

class Module:

    def zero_grad(self):
        for p in self.parameters():
            p.grad = 0

    def parameters(self):
        return []


class ReLU(Module):
    """
        Implementación de la clase ReLU como capa de la red neuronal
    """
    def __call__(self, x):
        if isinstance(x, list):
            return [xi.relu() for xi in x]
        return x.relu()


class Tanh(Module):
    """
        Implementación de la clase Tanh como capa de la red neuronal
    """
    def __call__(self, x):
        if isinstance(x, list):
            return [xi.tanh() for xi in x]
        return x.tanh()

class Sigmoid(Module):
    """
        Implementación de la clase Sigmoid como capa de la red neuronal
    """
    def __call__(self, x):
      if isinstance(x, list):
        return [xi.sigmoid() for xi in x]
      return (x.sigmoid())


class Neuron(Module):

    def __init__(self, nin):
        self.w = [Value(random.uniform(-1,1)) for _ in range(nin)]
        self.b = Value(0)
        self.nonlin = False # Por defecto queda lineal

    def __call__(self, x):
        act = sum((wi*xi for wi,xi in zip(self.w, x)), self.b)
        return act

    def parameters(self):
        return self.w + [self.b]

    def __repr__(self):
        return f"{'ReLU' if self.nonlin else 'Linear'}Neuron({len(self.w)})"

class Linear(Module):

    def __init__(self, nin, nout, **kwargs):
        self.neurons = [Neuron(nin) for _ in range(nout)]

    def __call__(self, x):
        out = [n(x) for n in self.neurons]
        return out[0] if len(out) == 1 else out

    def parameters(self):
        return [p for n in self.neurons for p in n.parameters()]

    def __repr__(self):
        return f"Layer of [{', '.join(str(n) for n in self.neurons)}]"

class MLP(Module):
    """
    """
    def __init__(self, nin, nouts, activations=[ReLU()]): # Hasta ahoara el modelo se instancia así: xor = MLP(2, [3,3,1]), pero ahora se busca colocar las capas de activación (Relu, Tanh, Sigmoid) también, osea: xor =  MLP (2, [3, 3, 1], activations=[ReLU(), Tanh(), ReLU()])
        sz = [nin] + nouts
        self.activations = activations
        self.act = 0
        self.layers = [Linear(sz[i], sz[i+1], nonlin=i!=len(nouts)-1) for i in range(len(nouts))]

    def __call__(self, x):
        for layer in self.layers:
            # Por cada capa hay que matchear la capa de activación correspondiente
            # Si la cantidad de capas de activación es 1, se usa siempre la misma en cambio si hay más de una se itera
            x = layer(x)
            if len(self.activations) == 1:
                 x = self.activations[0](x)
            else:
                x = self.activations[self.act](x)
                self.act += 1

        self.act = 0

        return x

    def parameters(self):
        return [p for layer in self.layers for p in layer.parameters()]

    def __repr__(self):
        return f"MLP of [{', '.join(str(layer) for layer in self.layers)}]"

# Generación del modelo usando las capas nuevas

In [184]:
np.random.seed(40)
activations = [Tanh()]
xor = MLP(2, [3, 3,1], activations=activations)
xs = [[0, 0], [0,1], [1,0], [1, 1]]
ys = [0, 1, 1, 0]
xor.parameters()

[Value(data=0.20103493592459598, grad=0, name=),
 Value(data=0.1296988923867568, grad=0, name=),
 Value(data=0, grad=0, name=),
 Value(data=-0.7941867022248725, grad=0, name=),
 Value(data=-0.9331314998939846, grad=0, name=),
 Value(data=0, grad=0, name=),
 Value(data=-0.5227004423305794, grad=0, name=),
 Value(data=-0.2143356975800632, grad=0, name=),
 Value(data=0, grad=0, name=),
 Value(data=0.16238916484311772, grad=0, name=),
 Value(data=-0.49293466967830657, grad=0, name=),
 Value(data=-0.6462843620423253, grad=0, name=),
 Value(data=0, grad=0, name=),
 Value(data=-0.5188355624655208, grad=0, name=),
 Value(data=-0.22438179330656416, grad=0, name=),
 Value(data=-0.9330008381018684, grad=0, name=),
 Value(data=0, grad=0, name=),
 Value(data=0.03388853383116386, grad=0, name=),
 Value(data=0.5149379874440994, grad=0, name=),
 Value(data=0.42693349280672965, grad=0, name=),
 Value(data=0, grad=0, name=),
 Value(data=0.28616927784108004, grad=0, name=),
 Value(data=0.7449652180012074

In [185]:
yhats = [xor(x) for x in xs]
yhats

[Value(data=0.0, grad=0, name=),
 Value(data=0.3973904042836106, grad=0, name=),
 Value(data=0.5284864672295643, grad=0, name=),
 Value(data=0.6213245639782784, grad=0, name=)]

# Entrenamiento del modelo MLP usando las capas





## Caso en que no le pasemos ninguna función de activación, por defecto queda ReLU()

In [255]:
np.random.seed(40)
#activations = [Tanh(), Tanh(), Tanh()]
xor = MLP(2, [3, 3,1])
xs = [[0,0], [0, 1], [1, 0], [1, 1]]
ys = [0, 1, 1, 0]
steps = 1000
learning_rate = 0.01

for _ in range(steps):
  # 1. forward pass
  yhats = [xor(x) for x in xs]
  # 2. calcular loss function
  loss = sum([(y - yhat)**2 for y, yhat in zip(ys, yhats)])/len(ys)
  # 3. zero gradient
  xor.zero_grad()
  # 4. backward pass
  loss.backward()
  # 5. update
  for p in xor.parameters():
    p.data -= p.grad * learning_rate

print(loss)

Value(data=5.1183517519314445e-05, grad=1, name=)


In [256]:
xor([0,0]), xor([0,1]), xor([1,0]), xor([1,1])

(Value(data=0.008979253280076906, grad=0, name=),
 Value(data=0.9958775506710758, grad=0, name=),
 Value(data=0.9950818953108729, grad=0, name=),
 Value(data=0.008973294291937595, grad=0, name=))

## Caso en que se usa Tahn

In [262]:
np.random.seed(40)
activations = [Tanh(), Tanh(), Tanh()]
xor = MLP(2, [3, 3,1], activations)
xs = [[0,0], [0, 1], [1, 0], [1, 1]]
ys = [0, 1, 1, 0]
steps = 1000
learning_rate = 0.01

for _ in range(steps):
  # 1. forward pass
  yhats = [xor(x) for x in xs]
  # 2. calcular loss function
  loss = sum([(y - yhat)**2 for y, yhat in zip(ys, yhats)])/len(ys)
  # 3. zero gradient
  xor.zero_grad()
  # 4. backward pass
  loss.backward()
  # 5. update
  for p in xor.parameters():
    p.data -= p.grad * learning_rate

print(loss)

Value(data=0.16770244050784894, grad=1, name=)


In [263]:
xor([0,0]), xor([0,1]), xor([1,0]), xor([1,1])

(Value(data=0.044405085521352354, grad=0, name=),
 Value(data=0.6330651640932666, grad=0, name=),
 Value(data=0.6621798150959687, grad=0, name=),
 Value(data=0.6480533099213205, grad=0, name=))

## Caso en que se quieran combinar las funciones de activacion

In [273]:
np.random.seed(40)
activations = [ReLU(), Tanh(), Tanh()]
xor = MLP(2, [3, 3,1], activations)
xs = [[0,0], [0, 1], [1, 0], [1, 1]]
ys = [0, 1, 1, 0]
steps = 1000
learning_rate = 0.01

for _ in range(steps):
  # 1. forward pass
  yhats = [xor(x) for x in xs]
  # 2. calcular loss function
  loss = sum([(y - yhat)**2 for y, yhat in zip(ys, yhats)])/len(ys)
  # 3. zero gradient
  xor.zero_grad()
  # 4. backward pass
  loss.backward()
  # 5. update
  for p in xor.parameters():
    p.data -= p.grad * learning_rate

print(loss)

Value(data=0.023376868271005647, grad=1, name=)


In [275]:
xor([0,0]), xor([0,1]), xor([1,0]), xor([1,1])

(Value(data=0.10155868558963255, grad=0, name=),
 Value(data=0.7980419946178411, grad=0, name=),
 Value(data=0.7966572180343555, grad=0, name=),
 Value(data=0.02738060387177022, grad=0, name=))